CPP Pricing Engine<br>
Alejandro Naranjo Caraza<br>
2026-03-01

#### Overview

In [6]:
from IPython.display import display, HTML

#### Binomial Model

<u>General Overview of Option Valuation</u>

The binomial model for option valuation proposes a multiplicative random walk as a model for stock price behaviour (see *insert link*). Under this model, the stock price at time $t+\Delta t$ is

$$
S_{t+\Delta t} =
\begin{cases}
u S_t & \text{with probability } p, \quad (u > 1) \\
d S_t & \text{with probability } 1-p, \quad (d < 1)
\end{cases}
$$

That is, if the stock moves up, its value becomes $S_u = u S_t$, and if it moves down, its value becomes $S_d = d S_t$.

An option with underlying stock price $S_t$ is considered. The option's payoff if the stock moves to $S_u$ is $f_u$. If the stock moves to $S_d$, the payoff is $f_d$. An investor can construct a riskless portfolio consisting of a long position in $\Delta$ shares of stock and a short position in one option.

The value of the portfolio at time $t+\Delta t$ is

$$
\begin{align*}
\Delta S_t u - f_u & \quad \text{if the stock price moves up} \\
\Delta S_t d - f_d & \quad \text{if the stock price moves down}
\end{align*}
$$

The cost of setting up the portfolio at time $t$ is
$$
\Delta S_t - f
$$

Notice that the following conditions hold:

- The portfolio is riskless if

$$
\Delta S_t u - f_u = \Delta S_t d - f_d
$$
$$
\Delta =
\frac{f_u - f_d}
{S_t (u - d)}
$$

- The current cost of setting up the portfolio is equal to the present value of the portfolio at time $t+\Delta t$ if

$$
e^{-r \, \Delta t} (\Delta S_t u - f_u) = e^{-r \, \Delta t} (\Delta S_t d - f_d) = \Delta S_t - f
$$
$$
f = e^{-r \Delta t} (p f_u + (1-p) f_d)
$$ 
<div style="margin-left: 2em;">
where $p = \dfrac{a - d}{u - d}$ is the risk-neutral probability, $a = e^{r \Delta t}$ is the risk-free growth factor, $\Delta t$ is the length of each time step, and $r$ is the risk-free interest rate.
</div>
<br>

<u>Interpretation of the Risk-Neutral Probability</u>

The probability $p$ used in the binomial pricing formula is called the *risk-neutral probability*. It does not represent the real-world probability that the stock price will move up.

Instead, this probability arises from the requirement that arbitrage opportunities must not exist. Because it is possible to construct a hedged portfolio consisting of the stock and the option that eliminates risk, the return on this portfolio must equal the risk-free rate. This condition uniquely determines the risk-neutral probability.

As a result, the option price can be interpreted as the discounted expected payoff computed using the risk-neutral probability. This does not imply that investors are risk-neutral in reality; rather, prices behave *as if* investors were risk-neutral because arbitrage prevents any alternative pricing.

In [39]:
#| echo: false

display(HTML("""
<div style="text-align: center;">
    <p style="font-size: 17px; text-align: center; font-weight: bold; margin-top: 8px;">
    <i>
       Example of a Binomial Tree
    </i>
    </p>
    <img src="./assets/binomialTree.png" width="500">
</div>
"""))

<u>Implementation Details</u>

From the equations derived above, the option price at time $t=t_0$ (present value) can be calculated by constructing a binomial tree with backward induction:

- First, calculate the stock prices and option payoffs at the final nodes, that is, at maturity ($t=T$).
- Move backwards through the tree. Consider the nodes at $t = T - \Delta t$ and calculate the option value using  
  $$
  f = e^{-r \Delta t} (p f_u + (1-p) f_d)
  $$
- Continue moving backwards through the binomial tree until reaching the node at $t=0$.

<u>Binomial Pricing Model for American Options</u>

American options can be exercised at any time prior to maturity $T$. Consequently, when working backwards through the binomial tree, the continuation value at each node is compared with the intrinsic value from immediate exercise. The option value $f$ at each node is the maximum of these two quantities.

<u>Binomial Pricing with Continuous Dividend Yield</u>

For a stock paying a continuous dividend yield $q$, the expected growth rate under the risk-neutral measure is $r - q$. Accordingly, the expected growth factor per time step is $a = e^{(r - q)\Delta t}$. This adjustment must be incorporated into the binomial pricing model, particularly when computing the risk-neutral probability.

<u>Binomial Pricing with Known Discrete Dividend Cash Schedule</u>

When an option's underlying stock has a known discrete dividend cash schedule, 
subtracting the dollar amount from the stock's value at each node at or after 
the ex-dividend dates results in a non-recombining tree (see *insert link*).

An alternative method is to separate the stock price $S_t$ into an uncertain 
component $S_t^*$ (used to model the *random walk*) and a certain component 
(the present value of future dividends paid during the life of the option).

For a stock with a single dividend payment $D$ and ex-dividend date $\tau$, $S^*$ is given by

$$
S^* =
\begin{cases}
S & \text{when } i\Delta t > \tau \\[6pt]
S - D e^{-r(\tau - i\Delta t)} & \text{when } i\Delta t \le \tau
\end{cases}
$$

For a set of dividend payments $\{D_j : j = 1,2,\ldots,k\}$ 
and corresponding ex-dividend dates $\{\tau_j : j = 1,2,\ldots,k\}$:

$$
S^* = S - \sum_{j:\,\tau_j \ge i\Delta t} 
D_j e^{-r(\tau_j - i\Delta t)}
$$

Using $\sigma^*$, the volatility of $S_t^*$, a recombining tree that models 
$S_t^*$ can be constructed as before. Adding the corresponding known component 
at each node results in a tree for $S_t$, and option valuation can then be 
carried out as before.

Note: $\sigma^* = \sigma$ is sometimes used for model simplicity, while $\sigma^* = \sigma\dfrac{S}{S^*}$ is a more accurate approximation.

#### Black-Scholes-Merton

From *insert link*, the Black-Scholes-Merton (BSM) assumes stock price follows a *Geometric Brownian Motion*:

$$
dS = \mu S dt + \sigma S dz
$$

If $f$ is the price of an option contingent on $S$, if it must be some function of $S$ and $t$: $f=f(S,t)$.<br>
From Ito's lemma (see *insert link*)

$$
df = (\dfrac{\partial f}{\partial S}\mu S + \dfrac{\partial f}{\partial t} + \dfrac{1}{2} \dfrac{\partial² f}{\partial S²}\sigma²S²)dt + \dfrac{\partial f}{\partial S} \sigma S dz
$$

Notice that the *Winer process* underlying $f$ and $S$ is the same. This means that a portfolio of the stock and the option can be constructed in such a way that the process is eliminated, therefore eliminating risk.

Taking a short position in one option and a long position in $\dfrac{\partial f}{\partial S}$ shares results in a portfolio of value

$$
\Pi = -f + \dfrac{\partial f}{\partial S}S
$$

Notice the following is true:

$$
d\Pi = -df + \dfrac{\partial f}{\partial S}dS + S d(\dfrac{\partial f}{\partial S})
$$

Since $\dfrac{\partial f}{\partial S}$ is held constant over $dt$:

$$
d\Pi = -df + \dfrac{\partial f}{\partial S}dS
$$

Substituting in the previous equations:

$$
d\Pi = (-\dfrac{\partial f}{\partial t} - \dfrac{1}{2}\dfrac{\partial² f}{\partial S²}\sigma² S²)dt
$$

Under no riskless arbitrage opportunities:

$$
d\Pi = r \Pi dt
$$
$$
(\dfrac{\partial f}{\partial t} + \dfrac{1}{2}\dfrac{\partial² f}{\partial S²}\sigma² S²)dt = r(f - \dfrac{\partial f}{\partial S}S)dt
$$
$$
\dfrac{\partial f}{\partial t} + rS\dfrac{\partial f}{\partial S} + \dfrac{1}{2}\sigma²S²\dfrac{\partial² f}{\partial S²} = rf
$$

is the Black-Scholes-Merton differential equation.<br>
The equation has the following analytic solution (see *insert link*):

$$
\begin{align*}
c &= S_0 N(d_1) - K e^{-rT} N(d_2) \\
p &= K e^{-rT} N(-d_2) - S_0 N(-d_1)
\end{align*}
$$

with

$$
\begin{align*}
d_1 &= \dfrac{\ln(S_0/K) + \left(r + \frac{\sigma^2}{2}\right)T}
{\sigma \sqrt{T}} \\
d_2 &= \dfrac{\ln(S_0/K) + \left(r - \frac{\sigma^2}{2}\right)T}
{\sigma \sqrt{T}}
= d_1 - \sigma \sqrt{T}
\end{align*}
$$

where

- $c$ is the call option's price
- $p$ is a put option's price
- $N(.)$ is the cumulative distribution function of a standard normal random variable.

<u>Implementation Details</u>

Since the Black–Scholes–Merton (BSM) differential equation admits a closed-form analytic solution under standard model assumptions, implementation of the model for option valuation reduces to a direct calculation given the required input variables.